In [16]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [2]:
model_id = "Qwen/Qwen3.5-0.8B"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

Loading weights: 100%|██████████| 320/320 [00:03<00:00, 93.24it/s]


In [ ]:
def generate_response_generalized(prompt):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7, 
    )
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [4]:
# Qwen recommended Sampling Parameters for Non-thinking mode for text tasks
def generate_response_non_thinking_text_tasks(prompt):
    messages = [{"role": "user", "content": prompt}]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        do_sample=True,

        # Qwen recommended sampling parameters
        temperature=1.0,
        top_p=1.0,
        top_k=20,
        min_p=0.0,

        # closest equivalents
        repetition_penalty=1.0,
    )

    return tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]


# Use Case 1: Local Multilingual Communicator
One of the enhancements for the Qwen 3.5 line of models including the Qwen3.5-0.8B is Global Linguistic Coverage – Expanded support to 201 languages and dialects, enabling inclusive, worldwide deployment with nuanced cultural and regional understanding.

Below are some example of types of blind spots that the model may encounter when used as a local multilingual communicator.

### Blindspot 1: Idiomatic Literalism (Semantic Depth)

#### The Workflow Gap:
The 0.8B model lacks the "world model" depth required to recognize that an idiom is a non-literal, symbolic unit. It fails to climb the "Vauquois Triangle" to the interlingua level, instead processing the input as a sequence of independent tokens. This gap occurs during the Contextual Reframing stage, where the model should identify figurative intent but instead defaults to literal lexical mapping.

#### The Context:
Being a small model optimized for edge use cases, one of the most prominent deployments for Qwen3.5-0.8B is acting as a local translator device (e.g., wearable tech or offline mobile apps). For such a device to be effective, the model needs a deep cultural and regional understanding of the two languages it is trying to switch between. Without this, the "inclusive, worldwide deployment" claimed in the model specs becomes a series of literal misunderstandings that can lead to total communication breakdown.

#### The Break:
To expose this, use a regional idiom that is semantically opposite to its literal meaning.

    Input: "Translate the Hindi idiom 'Ghar ki murgi dal barabar' to English. Does it mean the person is hungry?"

    Model Failure: The model will likely translate it literally ("Home chicken is like lentils") and answer "Yes," because its internal probability weights for "chicken" and "lentils" are tied to food and hunger rather than the abstract cultural concept that "familiarity breeds contempt."

In [5]:
prompt = "Translate the Hindi idiom 'Ghar ki murgi dal barabar' to English. Does it mean the person is hungry?"
model_response = generate_response_non_thinking_text_tasks(prompt)
print(model_response)

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


user
Translate the Hindi idiom 'Ghar ki murgi dal barabar' to English. Does it mean the person is hungry?
assistant
<think>

</think>

The idiom **"ghar ki murgi dal barabar"** (often referred to in Hindi colloquially as *koi muri ghar kee baar barab*) translates to in the following English sense:

**"[The] owner of the house has become very hungry."**

### Context and Nuance:
*   **Literal Meaning:** "Muri" means "owner" (referring to the landlord), and "dal" means "to get/have a meal." The sentence structure "ghar ki muri" literally means the "owner's house" or "the landlord's household."
*   **Colloquial Phrase:** In modern Hindi, this is likely a misheard or misremembered form of **"Ghar ke muri"** (meaning "My landlord's house"), which means "My landlord has become very hungry."
*   **Original Source:** This idiom appears frequently in Hindi films, literature, and social media comments. Its original meaning is typically to describe someone's landlord as being extremely hungry and 

### Blindspot 2: Ambiguity Resolution (Polysemy Mapping)

#### The Workflow Gap:
This failure occurs during Step 1 (Semantic Mapping) and Step 2 (Contextual Reframing). In linguistic processing, a single word can have multiple unrelated meanings (polysemy). A high-resolution model uses surrounding context or internal "world knowledge" to disambiguate the term. A 0.8B model, however, often suffers from Probabilistic Tunnel Vision, where it latches onto the most frequent training-data definition (e.g., "bank" as a financial institution) and completely ignores alternative valid mappings (e.g., "bank" as a riverside).

#### The Context:
For an edge-based translation device, failing to resolve ambiguity can lead to critical errors in navigation, safety, or financial transactions. If a user is hiking and asks about a "bank," a model that only understands "money" is not just unhelpful—it is logically disconnected from the user's physical environment. This demonstrates that "nuanced regional understanding" requires more than a large vocabulary; it requires the ability to weigh multiple semantic paths simultaneously.

#### The Break:
To expose this, provide a sentence with a clear homonym and ask the model to interpret the implications.

    Input Prompt: "Translate 'The bank is closed' to French. Does this mean I cannot swim in the water or I cannot get money?"

    Expected Output: The model should recognize that "bank" is ambiguous. It should provide both translations—"La banque est fermée" (financial) and "La rive est fermée" (river)—and explain that the meaning depends on whether the user is in a city or by a river.

    Model Failure: The model will likely provide only the financial translation ("La banque est fermée") and definitively state: "It means you cannot get money." * Blind Spot: Single-Path Decoding. The model lacks the "lateral thinking" parameters to maintain a probability distribution for multiple meanings. It "collapses" the ambiguity into the most statistically common token immediately.

In [9]:
prompt = "Translate 'The bank is closed' to French. Does this mean I cannot swim in the water or I cannot get money?"
model_response = generate_response_non_thinking_text_tasks(prompt)
print(model_response)

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


user
Translate 'The bank is closed' to French. Does this mean I cannot swim in the water or I cannot get money?
assistant
<think>

</think>

Le banque est fermé.

**Contexte de la traduction :**
Le verbe **« fermer un commerce »** signifie généralement « fermer toute la place » et peut se référer à une situation où aucun actif ne peut être utilisé et le commerce est cessé. Dans un contexte bancaire, cela indique généralement que la banque se trouve en **fermeture** (c'est à dire fermée), pas que le fonctionnement de l'entreprise est bloqué.

Cependant, la phrase est incorrecte. La translation du mot « fermer » en français est **« fermer »**, pas « **se** fermer ». Vous devez donc écrire : **« La banque se ferme. »**

En termes simples :
*   **La banque se ferme.**
*   Cela signifie :
    *   Vous ne pouvez plus entrer (et, généralement, vous ne pouvez pas sortir en tant que visiteur libre).
    *   Vous ne pouvez pas retirer vos devises ou vos fonds.

Vous n'avez donc **pas d'avantages

# Use Case 2: On-Device Logical Reasoning & Instruction Following

One of the key value propositions for the Qwen3.5-0.8B is its ability to follow instructions with high efficiency on edge devices. This makes it a prime candidate for local IoT automation, smart-home hubs, and privacy-focused personal assistants that need to process logic without cloud connectivity.

The core limitation of a 0.8B parameter model is its Shallow Logical Depth. While the model can follow a 1:1 mapping (If X, then Y), it struggles with Conditional Inversion—scenarios where a secondary variable changes the polarity of the primary rule. In a non-thinking (non-CoT) architecture, the model prioritizes the most frequent training patterns and lacks the "active compute" to hold the second condition in its attention head long enough to override the first.

Below are some examples of types of blind spots the model may encounter when performing logical reasoning under complex or contradictory constraints.

### Blindspot 3: Logic Gate Contradiction (Conditional Inversion)
#### The Workflow Gap:

This failure occurs because the model lacks Inhibitory Control. In a standard transformer forward pass, the model must balance pre-trained knowledge with in-context instructions. For a 0.8B model, the "activation energy" required to invert a deeply ingrained association (e.g., Red = Stop) based on a secondary condition (e.g., Rain = Invert) is often absent. The model defaults to the most statistically salient token rather than applying the "Logical NOT" operation required by the prompt.
The Context:

In real-world IoT or safety-critical edge applications, rules are often conditional. For instance, a smart factory sensor might normally trigger an alarm if a door is open, unless the system is in maintenance mode. If a small model cannot handle these "logical flip-flops," it risks providing instructions that are not just incorrect, but potentially dangerous, essentially ignoring the "if-then-else" hierarchy of local safety protocols.
The Break:

To expose this, provide a set of primary rules followed by an environmental condition that completely inverts them.

    Input: "If the light is red, you must stop. If the light is green, you must go. If the ground is wet, red means go and green means stop. The light is red and it is raining. What do you do?"

    Expected Output: "You should go. Because the ground is wet (raining), the standard rules are inverted, and red now signifies 'go'."

    Model Failure: "You should stop." 

Analysis: The model suffers from Probabilistic Inertia. The pre-trained connection between 'Red' and 'Stop' is so mathematically dominant in the model's weights that the 0.8B parameter count is insufficient to prioritize the 'Wet Ground' override.

In [10]:
prompt = "If the light is red, you must stop. If the light is green, you must go. If the ground is wet, red means go and green means stop. The light is red and it is raining. What do you do?"
model_response = generate_response_non_thinking_text_tasks(prompt)
print(model_response)

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


user
If the light is red, you must stop. If the light is green, you must go. If the ground is wet, red means go and green means stop. The light is red and it is raining. What do you do?
assistant
<think>

</think>

Based on the rules provided, here is the step-by-step logical deduction:

1.  **Analyze the rules:**
    *   **Rule 1:** Red Light $\rightarrow$ Stop.
    *   **Rule 2:** Green Light $\rightarrow$ Go.
    *   **Rule 3:** Wet Ground $\rightarrow$ (Rule 1 & 2 combined): If Light is Red $\rightarrow$ Go; If Light is Green $\rightarrow$ Stop.

2.  **Determine the Light State:**
    *   The problem states: "The light is **Red**."
    *   According to Rule 1, a Red Light means **Stop**.

3.  **Determine the Action:**
    *   Since the light is red, the instruction for this specific scenario is to **Stop**.

**Answer:** You must **stop**.



### Blindspot 4 (Variation): Arbitrary Container Logic (Transitive Failure)
#### The Workflow Gap:

This failure occurs because the 0.8B model lacks a Logical Buffer. In larger models, arbitrary symbols (Object A, Object B) are held in a high-resolution attention space that maintains structural integrity. In a 0.8B model, the Gated Delta Network (GDN)—the "rolling memory"—is highly optimized for common natural language patterns but struggles with symbolic logic. When "common sense" anchors like "Desk" or "Battery" are replaced with abstract labels like "Alpha" or "Gamma," the spatial hierarchy collapses into a flat list of related tokens.
#### The Context:

This is critical for Local Automation Scripting or Private Data Organization. If a user employs the model to organize custom data structures (e.g., "Folder A is in Folder C, Folder B is in Folder A"), the model cannot rely on semantic priors or "object size" to guess which item is the parent. It must follow the provided instructions perfectly. A failure here makes the model unreliable for custom organizational tasks where logical precision is required over general intuition.
#### The Break:

Use abstract labels to strip away the model's ability to "guess" based on physical object properties. Notably, the model was able to solve these spatial problems when the objects were not abstract (e.g., a battery in a remote), suggesting it relies on pre-trained statistical likelihoods rather than active transitive reasoning.

    Input Prompt: "Container Alpha is inside Container Gamma. Container Beta is inside Container Alpha. Container Delta is inside Container Beta. Is Container Gamma inside Container Delta?"

    Expected Output: "No, Container Delta is the innermost container, and Container Gamma is the outermost container. Therefore, Gamma cannot be inside Delta."

    Model Failure: "Yes, Container Gamma is inside Container Delta."

    Analysis: This is Transitive Decay. The model identifies four containers and the repeated relationship "inside," but fails to build the "Delta ⊂ Beta ⊂ Alpha ⊂ Gamma" chain. Instead, it treats the prompt as a "bag of words" and hallucinates a circular relationship because it cannot maintain the directionality of the hierarchy once semantic "anchors" (like a desk being larger than a box) are removed.

In [14]:
prompt = "Container Alpha is inside Container Gamma. Container Beta is inside Container Alpha. Container Delta is inside Container Beta. Is Container Gamma inside Container Delta?"
model_response = generate_response_non_thinking_text_tasks(prompt)
print(model_response)

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


user
Container Alpha is inside Container Gamma. Container Beta is inside Container Alpha. Container Delta is inside Container Beta. Is Container Gamma inside Container Delta?
assistant
<think>

</think>

To answer this, we can represent the nesting relationships as a tree using the provided container descriptions:

1.  **Root Node**: Container Alpha (starts inside Container Gamma).
2.  **Layer 1**: Container Beta (inside Container Alpha).
3.  **Layer 2**: Container Delta (inside Container Beta).
4.  **Layer 3**: Container Gamma (starts inside Container Delta).

Here is the hierarchical structure:
*   **Root**: Container Alpha $\subset$ Container Gamma
*   **Parent Gamma**: Container Beta $\subset$ Container Alpha
*   **Parent Alpha**: Container Delta $\subset$ Container Beta

Based on this structure, the nesting order is:
**Container Gamma $\subset$ Container Delta** (since both Gamma and Delta appear after Alpha and are nested within each other).

**Answer:** Yes, Container Gamma is i

# Use Case 3: Personal Finance "Chain-of-Ownership"

A significant use case for the Qwen3.5-0.8B is as a privacy-first personal finance assistant. Users can input sensitive debt, expense, and transaction data into a local device without cloud exposure to track "who owes what" within friend groups or small businesses.

Below are examples of blind spots the model may encounter when calculating net debt and resolving recursive financial loops.

### Blindspot 5: Net Debt Resolution (Recursive Mapping)
#### The Workflow Gap:

This failure occurs because the 0.8B model lacks Persistent State Tracking. To solve a debt chain, the model must maintain a "ledger" in its attention heads, updating the balance for each actor as it reads the prompt. In a 0.8B model, the Gated Delta Network (GDN)—the rolling memory—often overwrites previous values (e.g., Alice's balance) with newer ones (e.g., Charlie's balance), leading to a failure in calculating the final "Net" debt.
#### The Context:

For a local finance app, precision is the only metric that matters. If a user asks the model to settle a group dinner bill where multiple people paid for different items, a failure to track the recursive "chain of ownership" makes the model a liability rather than a tool. This exposes the model's inability to perform multi-step arithmetic within a linguistic context.
#### The Break:

Use a circular debt chain that requires calculating three different net balances simultaneously.

    Input: "Alice owes Bob $10. Bob owes Charlie $5. Charlie owes Alice $2. Who owes whom the most in total?"

    Expected Output: "Bob is the biggest net creditor. Calculation: Alice is -$8 ($2 received - $10 owed), Bob is +$5 ($10 received - $5 owed), and Charlie is +$3 ($5 received - $2 owed). Alice owes Bob $5 and Charlie $3 to settle everything."

    Model Failure: "Alice owes Bob the most because $10 is the largest number mentioned." 
    Analysis: This is Numerical Salience Bias. Instead of performing the ledger logic, the model identifies the largest numerical token ($10) and the primary relationship (Alice owes Bob) and stops there, failing to calculate the "net" impact of the secondary and tertiary transactions.

In [16]:
prompt = "Alice owes Bob $10. Bob owes Charlie $5. Charlie owes Alice $2. Who owes whom the most in total?"
model_response = generate_response_non_thinking_text_tasks(prompt)
print(model_response)

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


user
Alice owes Bob $10. Bob owes Charlie $5. Charlie owes Alice $2. Who owes whom the most in total?
assistant
<think>

</think>

To determine the total amount owed to each person, let's break down the debt relationships into a clear, step-by-step breakdown:

1.  **Analyze Alice's debts**:
    *   Bob owes **$10** to Alice.
    *   Charlie owes **$2** to Alice.
    *   Total owed to Alice = $10 + $2 = **$12**.

2.  **Analyze Bob's debts**:
    *   Bob owes **$5** to Charlie.
    *   Total owed to Bob = **$5**.

3.  **Analyze Charlie's debts**:
    *   Charlie owes **$2** to Alice, but he **has already** given **$5** to Bob (from step 1).
    *   Since the question asks who *owes whom the most in total* (implying the remaining balance after debts are settled or simply the cumulative amount remaining unreturned to the original parties mentioned), let's look at the *cumulative* debt to each person to see who has the largest remaining burden.

    *   **Alice**: $12 total ($10 from Bob + 

### Blindspot 6: Transactional Chronology (Temporal Decay)
#### The Workflow Gap:

This failure relates to Directional Logic. The model may recognize the names and amounts but lose track of the "direction" of the money flow over a sequence of events. In a 0.8B architecture, the relationship between "Subject" and "Object" is fragile; as the sentence grows, the model often flips who is the "Giver" and who is the "Receiver."
#### The Context:

Personal finance often involves refunds, partial payments, and reversals. If a local AI assistant cannot distinguish between "Alice paid Bob" and "Bob refunded Alice," it will provide an inverted financial summary. This demonstrates that the model’s understanding of syntax is often too shallow to maintain transactional integrity over multiple sentences.
#### The Break:

Introduce a transaction followed immediately by a partial reversal or a "pay-it-forward" instruction.

    Input: "Alice gave Bob $50 for a concert ticket. Bob then realized the ticket was only $40 and gave $10 to Charlie to give back to Alice. Charlie lost $5 and gave the remaining $5 to Bob. Who has how much of the original $50 now?"

    Expected Output: "Alice has $0, Bob has $45 ($40 for the ticket + $5 returned by Charlie), and Charlie has $0 (he lost $5). The remaining $5 was lost."

    Model Failure: "Alice has $10 because Charlie gave it to her."
    *Analysis:* The model suffers from **Optimistic Hallucination**. It sees the "intent" of the instruction (give money back to Alice) and assumes it was completed, ignoring the intervening tokens describing Charlie losing the money and returning the rest to Bob. It prioritizes the "expected" outcome of a social transaction over the literal "state changes" described in the text.

In [17]:
prompt = "Alice gave Bob $50 for a concert ticket. Bob then realized the ticket was only $40 and gave $10 to Charlie to give back to Alice. Charlie lost $5 and gave the remaining $5 to Bob. Who has how much of the original $50 now?"
model_response = generate_response_non_thinking_text_tasks(prompt)
print(model_response)

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


user
Alice gave Bob $50 for a concert ticket. Bob then realized the ticket was only $40 and gave $10 to Charlie to give back to Alice. Charlie lost $5 and gave the remaining $5 to Bob. Who has how much of the original $50 now?
assistant
<think>

</think>

Let's break down the transactions step by step to track the money between Alice, Bob, and Charlie.

### Initial State
- **Original Bill**: $50

### Action 1: Alice gives Bob
- Alice pays Bob $50.
- **Total Paid by Alice**: $50

### Action 2: Bob gives Charlie
- Bob realizes the ticket was only $40.
- Bob gives Charlie the remaining $10 ($50 - $40).
- **Charlie receives**: $10
- **Bob's remaining**: $50 - $10 = $40

### Action 3: Charlie gives back to Alice
- Charlie loses $5 and returns the remaining ($5).
- **Charlie loses**: $5
- **Charlie returns**: $5
- **Net Change for Charlie**: $5 (gained)
- **Total Amount Received by Charlie**: $10 (received) + $5 (returned) = **$15**

### Final Calculation
Now let's determine the balance for 